# 02_classical_models

Перенос исходных ячеек, исторические результаты validation. Черновик: зависимости и повторный запуск ещё не проверены.

[Colab — основной источник](https://colab.research.google.com/github/SamDerden/SpeechSense/blob/main/%22ru_sentiment_pipeline_adapted_gzip_robust_ipynb%22.ipynb)

[Colab — нейросетевой источник](https://colab.research.google.com/github/SamDerden/SpeechSense/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22ru_sentiment_pipeline_adapted_gzip_robust_ipynb%22.ipynb)

Ссылки открывают загруженные исходники в Colab; исходные Drive URL требуют подтверждения.

---
## Train / Validation split

Используем готовое разбиение из CSV: `train` для обучения, `validation` для оценки. Validation не участвует ни в выборе обучающих примеров, ни в обучении векторизаторов.

In [ ]:
# Для полного эксперимента используем всю train-часть.
# Для более быстрого пробного запуска можно установить USE_FULL_TRAIN = False.
USE_FULL_TRAIN = True
MAX_TRAIN_SAMPLES = 60_000

available_splits = set(df_clean["split"].astype(str).str.lower().unique())
if {"train", "validation"}.issubset(available_splits):
    train_df = df_clean[df_clean["split"].astype(str).str.lower().eq("train")].copy()
    test_df = df_clean[df_clean["split"].astype(str).str.lower().eq("validation")].copy()
    print("Используется исходное разбиение из CSV: train / validation.")
else:
    # Резервный вариант на случай, если в файле нет готового validation.
    print("Внимание: train/validation не найдены. Создаю stratified train/test split 80/20.")
    train_df, test_df = train_test_split(
        df_clean, test_size=0.2, random_state=RANDOM_STATE,
        stratify=df_clean["sentiment"]
    )

if not USE_FULL_TRAIN and len(train_df) > MAX_TRAIN_SAMPLES:
    n_classes = train_df["sentiment"].nunique()
    samples_per_class = max(1, MAX_TRAIN_SAMPLES // n_classes)
    train_df = (
        train_df.groupby("sentiment", group_keys=False)
        .apply(lambda group: group.sample(
            n=min(len(group), samples_per_class), random_state=RANDOM_STATE
        ))
        .reset_index(drop=True)
    )
    print(f"Для ускорения взята стратифицированная подвыборка train: {len(train_df):,} строк.")
else:
    print(f"Используем train целиком: {len(train_df):,} строк.")

if train_df["sentiment"].nunique() < 2 or test_df["sentiment"].nunique() < 2:
    raise ValueError("В train или validation недостаточно классов для обучения и оценки.")

print("\nTrain labels:")
print(train_df["sentiment"].map(CLASS_MAP).value_counts())
print("\nValidation labels:")
print(test_df["sentiment"].map(CLASS_MAP).value_counts())


In [ ]:
X_train = train_df["text_clean"]
y_train = train_df["sentiment"]
X_test = test_df["text_clean"]
y_test = test_df["sentiment"]

print(f"Train: {len(X_train):,} | Validation: {len(X_test):,}")
print("Распределение в train:")
print(y_train.value_counts().sort_index().rename(index=CLASS_MAP))
print("\nРаспределение в validation:")
print(y_test.value_counts().sort_index().rename(index=CLASS_MAP))


---
## 2–3. Векторизации + классические модели

Обучаем несколько комбинаций «векторизатор + классификатор» на одном и том же train/test.


In [ ]:
# ---------- Модели ----------
models = {
    "MultinomialNB (BoW 1-2)": Pipeline([
        ("vect", CountVectorizer(ngram_range=(1, 2), min_df=3, max_features=100_000)),
        ("clf", MultinomialNB())
    ]),

    "BernoulliNB (бинарный BoW 1-2)": Pipeline([
        ("vect", CountVectorizer(ngram_range=(1, 2), binary=True, min_df=3, max_features=100_000)),
        ("clf", BernoulliNB())
    ]),

    "ComplementNB (TF-IDF 1-2)": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=100_000)),
        ("clf", ComplementNB())
    ]),

    "LogReg (TF-IDF 1-2)": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=100_000)),
        ("clf", LogisticRegression(max_iter=1000, C=1.0, n_jobs=-1, random_state=RANDOM_STATE))
    ]),

    "LinearSVC (TF-IDF 1-2)": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=100_000)),
        ("clf", LinearSVC(C=1.0, dual="auto", random_state=RANDOM_STATE, max_iter=2000))
    ]),

    "LinearSVC (word+char)": Pipeline([
        ("features", FeatureUnion([
            ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=50_000)),
            ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_features=30_000))
        ])),
        ("clf", LinearSVC(C=0.5, dual="auto", random_state=RANDOM_STATE, max_iter=2000))
    ]),
}

print(f"Будет обучено {len(models)} моделей\n")


In [ ]:
LEADERBOARD.clear()  # не дублируем строки, если обучение запускается повторно

# ---------- Обучение и оценка ----------
for name, model in models.items():
    print(f"Обучаю: {name} ...", end=" ", flush=True)
    t0 = time.time()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    elapsed = time.time() - t0
    report(name, y_test, y_pred, elapsed)
    print()


---
## 4. Итоговое сравнение


In [ ]:
df_lb = pd.DataFrame(LEADERBOARD).sort_values("macro-F1", ascending=False).reset_index(drop=True)

styled = (
    df_lb.style
    .format({
        "macro-F1": "{:.4f}",
        "accuracy": "{:.4f}",
        "обучение, с": "{:.1f}"
    })
    .highlight_max(subset=["macro-F1", "accuracy"], color="#d4edda")
    .set_caption("Leaderboard: macro-F1 и Accuracy на тестовой выборке")
    .set_table_styles([
        {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "15px"), ("font-weight", "bold")]},
        {"selector": "th", "props": [("text-align", "center"), ("background-color", "#f8f9fa")]},
        {"selector": "td", "props": [("text-align", "center")]}
    ])
    .hide(axis="index")
)

display(styled)


In [ ]:
# Барплот для наглядности
fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette("viridis", len(df_lb))
bars = ax.barh(df_lb["модель"], df_lb["macro-F1"], color=colors)
ax.set_xlabel("macro-F1")
ax.set_title("Сравнение моделей по macro-F1")
ax.set_xlim(0, 1)
for bar, val in zip(bars, df_lb["macro-F1"]):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f"{val:.4f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()


### 4.1 Подробный отчёт лучшей модели


In [ ]:
# Берём лучшую модель по macro-F1 на validation
if not LEADERBOARD:
    raise RuntimeError("Сначала выполните ячейку обучения моделей (ячейка 23).")

df_lb = pd.DataFrame(LEADERBOARD).sort_values("macro-F1", ascending=False).reset_index(drop=True)
best_name = df_lb.iloc[0]["модель"]
best_model = models[best_name]

print(f"Лучшая модель: {best_name}")
print(f"macro-F1 = {df_lb.iloc[0]['macro-F1']:.4f}, accuracy = {df_lb.iloc[0]['accuracy']:.4f}\n")

y_pred_best = best_model.predict(X_test)
print(classification_report(
    y_test, y_pred_best,
    labels=CLASS_ORDER,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))


In [ ]:
# Матрица ошибок на validation
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_best, labels=CLASS_ORDER)
disp = ConfusionMatrixDisplay(cm, display_labels=["Нейтр.", "Поз.", "Нег."])
disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
ax.set_title(f"Матрица ошибок — {best_name}")
plt.tight_layout()
plt.show()


### Как интерпретировать матрицу ошибок

Строки матрицы соответствуют истинной тональности, столбцы — предсказанной. Значения на диагонали — правильно классифицированные примеры; значения вне диагонали показывают, какие классы модель путает между собой. Конкретные числа формируются при запуске ноутбука и зависят от обученных моделей.

---
## Краткие выводы по фактическому запуску

Таблица и вывод ниже формируются по результатам текущего запуска на validation из предоставленного CSV. Фиксированные метрики из исходного ноутбука удалены, потому что они относятся к другому разбиению данных.


In [ ]:
# Метрики формируются динамически после обучения моделей.
df_lb = pd.DataFrame(LEADERBOARD).sort_values("macro-F1", ascending=False).reset_index(drop=True)
if df_lb.empty:
    print("Результатов пока нет. Сначала выполните ячейку обучения моделей.")
else:
    display(df_lb)
    best = df_lb.iloc[0]
    print(f"Лучшая модель по macro-F1: {best['модель']}")
    print(f"macro-F1: {best['macro-F1']:.4f}; accuracy: {best['accuracy']:.4f}")
    if len(df_lb) > 1:
        print(f"Разница macro-F1 с моделью на втором месте: {best['macro-F1'] - df_lb.iloc[1]['macro-F1']:.4f}")

print("\nУчитывайте, что train содержит несколько источников, включая 27 136 примеров с эвристической разметкой по рейтингу, "
      "а validation сформирован из исходного ru_sentiment_dataset_validation.")


---
## Train / Validation split

Используем готовое разбиение из CSV: `train` для обучения, `validation` для оценки. Validation не участвует ни в выборе обучающих примеров, ни в обучении векторизаторов.

In [ ]:
# Для полного эксперимента используем всю train-часть.
# Для более быстрого пробного запуска можно установить USE_FULL_TRAIN = False.
USE_FULL_TRAIN = True
MAX_TRAIN_SAMPLES = 60_000

available_splits = set(df_clean["split"].astype(str).str.lower().unique())
if {"train", "validation"}.issubset(available_splits):
    train_df = df_clean[df_clean["split"].astype(str).str.lower().eq("train")].copy()
    test_df = df_clean[df_clean["split"].astype(str).str.lower().eq("validation")].copy()
    print("Используется исходное разбиение из CSV: train / validation.")
else:
    # Резервный вариант на случай, если в файле нет готового validation.
    print("Внимание: train/validation не найдены. Создаю stratified train/test split 80/20.")
    train_df, test_df = train_test_split(
        df_clean, test_size=0.2, random_state=RANDOM_STATE,
        stratify=df_clean["sentiment"]
    )

if not USE_FULL_TRAIN and len(train_df) > MAX_TRAIN_SAMPLES:
    n_classes = train_df["sentiment"].nunique()
    samples_per_class = max(1, MAX_TRAIN_SAMPLES // n_classes)
    train_df = (
        train_df.groupby("sentiment", group_keys=False)
        .apply(lambda group: group.sample(
            n=min(len(group), samples_per_class), random_state=RANDOM_STATE
        ))
        .reset_index(drop=True)
    )
    print(f"Для ускорения взята стратифицированная подвыборка train: {len(train_df):,} строк.")
else:
    print(f"Используем train целиком: {len(train_df):,} строк.")

if train_df["sentiment"].nunique() < 2 or test_df["sentiment"].nunique() < 2:
    raise ValueError("В train или validation недостаточно классов для обучения и оценки.")

print("\nTrain labels:")
print(train_df["sentiment"].map(CLASS_MAP).value_counts())
print("\nValidation labels:")
print(test_df["sentiment"].map(CLASS_MAP).value_counts())


In [ ]:
X_train = train_df["text_clean"]
y_train = train_df["sentiment"]
X_test = test_df["text_clean"]
y_test = test_df["sentiment"]

print(f"Train: {len(X_train):,} | Validation: {len(X_test):,}")
print("Распределение в train:")
print(y_train.value_counts().sort_index().rename(index=CLASS_MAP))
print("\nРаспределение в validation:")
print(y_test.value_counts().sort_index().rename(index=CLASS_MAP))


---
## 2–3. Векторизации + классические модели

Обучаем несколько комбинаций «векторизатор + классификатор» на одном и том же train/test.


In [ ]:
# ---------- Модели ----------
models = {
    "MultinomialNB (BoW 1-2)": Pipeline([
        ("vect", CountVectorizer(ngram_range=(1, 2), min_df=3, max_features=100_000)),
        ("clf", MultinomialNB())
    ]),

    "BernoulliNB (бинарный BoW 1-2)": Pipeline([
        ("vect", CountVectorizer(ngram_range=(1, 2), binary=True, min_df=3, max_features=100_000)),
        ("clf", BernoulliNB())
    ]),

    "ComplementNB (TF-IDF 1-2)": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=100_000)),
        ("clf", ComplementNB())
    ]),

    "LogReg (TF-IDF 1-2)": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=100_000)),
        ("clf", LogisticRegression(max_iter=1000, C=1.0, n_jobs=-1, random_state=RANDOM_STATE))
    ]),

    "LinearSVC (TF-IDF 1-2)": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=100_000)),
        ("clf", LinearSVC(C=1.0, dual="auto", random_state=RANDOM_STATE, max_iter=2000))
    ]),

    "LinearSVC (word+char)": Pipeline([
        ("features", FeatureUnion([
            ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=50_000)),
            ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_features=30_000))
        ])),
        ("clf", LinearSVC(C=0.5, dual="auto", random_state=RANDOM_STATE, max_iter=2000))
    ]),
}

print(f"Будет обучено {len(models)} моделей\n")


In [ ]:
LEADERBOARD.clear()  # не дублируем строки, если обучение запускается повторно

# ---------- Обучение и оценка ----------
for name, model in models.items():
    print(f"Обучаю: {name} ...", end=" ", flush=True)
    t0 = time.time()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    elapsed = time.time() - t0
    report(name, y_test, y_pred, elapsed)
    print()


---
## 4. Итоговое сравнение


In [ ]:
df_lb = pd.DataFrame(LEADERBOARD).sort_values("macro-F1", ascending=False).reset_index(drop=True)

styled = (
    df_lb.style
    .format({
        "macro-F1": "{:.4f}",
        "accuracy": "{:.4f}",
        "обучение, с": "{:.1f}"
    })
    .highlight_max(subset=["macro-F1", "accuracy"], color="#d4edda")
    .set_caption("Leaderboard: macro-F1 и Accuracy на тестовой выборке")
    .set_table_styles([
        {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "15px"), ("font-weight", "bold")]},
        {"selector": "th", "props": [("text-align", "center"), ("background-color", "#f8f9fa")]},
        {"selector": "td", "props": [("text-align", "center")]}
    ])
    .hide(axis="index")
)

display(styled)


In [ ]:
# Барплот для наглядности
fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette("viridis", len(df_lb))
bars = ax.barh(df_lb["модель"], df_lb["macro-F1"], color=colors)
ax.set_xlabel("macro-F1")
ax.set_title("Сравнение моделей по macro-F1")
ax.set_xlim(0, 1)
for bar, val in zip(bars, df_lb["macro-F1"]):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f"{val:.4f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()


### 4.1 Подробный отчёт лучшей модели


In [ ]:
# Берём лучшую модель по macro-F1 на validation
if not LEADERBOARD:
    raise RuntimeError("Сначала выполните ячейку обучения моделей (ячейка 23).")

df_lb = pd.DataFrame(LEADERBOARD).sort_values("macro-F1", ascending=False).reset_index(drop=True)
best_name = df_lb.iloc[0]["модель"]
best_model = models[best_name]

print(f"Лучшая модель: {best_name}")
print(f"macro-F1 = {df_lb.iloc[0]['macro-F1']:.4f}, accuracy = {df_lb.iloc[0]['accuracy']:.4f}\n")

y_pred_best = best_model.predict(X_test)
print(classification_report(
    y_test, y_pred_best,
    labels=CLASS_ORDER,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))


In [ ]:
# Матрица ошибок на validation
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_best, labels=CLASS_ORDER)
disp = ConfusionMatrixDisplay(cm, display_labels=["Нейтр.", "Поз.", "Нег."])
disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
ax.set_title(f"Матрица ошибок — {best_name}")
plt.tight_layout()
plt.show()


### Как интерпретировать матрицу ошибок

Строки матрицы соответствуют истинной тональности, столбцы — предсказанной. Значения на диагонали — правильно классифицированные примеры; значения вне диагонали показывают, какие классы модель путает между собой. Конкретные числа формируются при запуске ноутбука и зависят от обученных моделей.

---
## Краткие выводы по фактическому запуску

Таблица и вывод ниже формируются по результатам текущего запуска на validation из предоставленного CSV. Фиксированные метрики из исходного ноутбука удалены, потому что они относятся к другому разбиению данных.


In [ ]:
# Метрики формируются динамически после обучения моделей.
df_lb = pd.DataFrame(LEADERBOARD).sort_values("macro-F1", ascending=False).reset_index(drop=True)
if df_lb.empty:
    print("Результатов пока нет. Сначала выполните ячейку обучения моделей.")
else:
    display(df_lb)
    best = df_lb.iloc[0]
    print(f"Лучшая модель по macro-F1: {best['модель']}")
    print(f"macro-F1: {best['macro-F1']:.4f}; accuracy: {best['accuracy']:.4f}")
    if len(df_lb) > 1:
        print(f"Разница macro-F1 с моделью на втором месте: {best['macro-F1'] - df_lb.iloc[1]['macro-F1']:.4f}")

print("\nУчитывайте, что train содержит несколько источников, включая 27 136 примеров с эвристической разметкой по рейтингу, "
      "а validation сформирован из исходного ru_sentiment_dataset_validation.")
